In [2]:

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text

from nhs_waiting_lists.constants import proj_db_path
from nhs_waiting_lists.utils.proj_paths import find_project_root

project_root = find_project_root()



In [3]:


DB_PATH = project_root / proj_db_path / "nhs_rttwtd.db"
DATA_DIR = "./data"

conn = create_engine(f"sqlite:///{DB_PATH}")


In [4]:


query = text("""
             SELECT provider_org_code                      AS provider,
                    treatment_function_code                AS treatment,
                    patients_with_unknown_clock_start_date AS unknown_clock_start,
                    *
             FROM all_rtt
             ORDER BY provider ASC, treatment ASC, period ASC; \
             """)

df = pd.read_sql(
    query,
    conn,
)
df

,provider,treatment,unknown_clock_start,period,provider_org_code,rtt_part_type,treatment_function_code,gt_00_to_01_weeks,gt_01_to_02_weeks,gt_02_to_03_weeks,...,gt_98_to_99_weeks,gt_99_to_100_weeks,gt_100_to_101_weeks,gt_101_to_102_weeks,gt_102_to_103_weeks,gt_103_to_104_weeks,gt_104_weeks,patients_with_unknown_clock_start_date,total,total_all
0,8KL73,C_301,0,2024-06,8KL73,Part_1A,C_301,30,49,13,...,0,0,0,0,0,0,0,0,101,101
1,8KL73,C_301,0,2024-06,8KL73,Part_2,C_301,35,23,8,...,0,0,0,0,0,0,0,0,0,70
2,8KL73,C_301,0,2024-06,8KL73,Part_2A,C_301,35,23,8,...,0,0,0,0,0,0,0,0,0,70
3,8KL73,C_301,0,2024-06,8KL73,Part_3,C_301,0,0,0,...,0,0,0,0,0,0,0,0,0,139
4,8KL73,C_301,0,2024-07,8KL73,Part_1A,C_301,20,68,20,...,0,0,0,0,0,0,0,0,130,130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
969801,Z9Z1G,C_999,0,2025-08,Z9Z1G,Part_1A,C_999,24,54,56,...,0,0,0,0,0,0,0,0,218,218
969802,Z9Z1G,C_999,0,2025-08,Z9Z1G,Part_1B,C_999,11,14,5,...,0,0,0,0,0,0,0,0,43,43
969803,Z9Z1G,C_999,0,2025-08,Z9Z1G,Part_2,C_999,22,24,17,...,0,0,0,0,0,0,0,0,0,118
969804,Z9Z1G,C_999,0,2025-08,Z9Z1G,Part_2A,C_999,17,16,11,...,0,0,0,0,0,0,0,0,0,78


In [5]:

from nhs_waiting_lists.constants import wait_ranges_gte_18, wait_ranges_lt_18

df["wait_lt_18"] = df[wait_ranges_lt_18].sum(axis=1, skipna=False)
df["wait_gte_18"] = df[wait_ranges_gte_18].sum(axis=1, skipna=False)
df["wait_sum"] = df["wait_lt_18"] + df["wait_gte_18"]
df["wait_pct_lt_18"] = df["wait_lt_18"] / df["wait_sum"]
df["wait_diff"] = df["wait_sum"] - df["total_all"]

df.query("provider == 'RAJ' and treatment == 'C_320' and period == '2025-08'")

,provider,treatment,unknown_clock_start,period,provider_org_code,rtt_part_type,treatment_function_code,gt_00_to_01_weeks,gt_01_to_02_weeks,gt_02_to_03_weeks,...,gt_103_to_104_weeks,gt_104_weeks,patients_with_unknown_clock_start_date,total,total_all,wait_lt_18,wait_gte_18,wait_sum,wait_pct_lt_18,wait_diff
388562,RAJ,C_320,0,2025-08,RAJ,Part_1A,C_320,17,9,13,...,0,6,0,239,239,182,57,239,0.761506,0
388563,RAJ,C_320,0,2025-08,RAJ,Part_1B,C_320,46,36,39,...,0,10,0,1209,1209,692,517,1209,0.572374,0
388564,RAJ,C_320,0,2025-08,RAJ,Part_2,C_320,439,525,622,...,0,0,0,0,9851,7213,2638,9851,0.732210,0
388565,RAJ,C_320,0,2025-08,RAJ,Part_2A,C_320,28,38,50,...,0,0,0,0,639,470,169,639,0.735524,0
388566,RAJ,C_320,0,2025-08,RAJ,Part_3,C_320,0,0,0,...,0,0,0,0,2489,0,0,0,NaN,-2489


In [11]:


# 2. Pivot to wide form
df_wide = (
    df.copy()
    .assign(
        rtt_part_type=lambda d: d["rtt_part_type"].map({
            "Part_1A": "admitted",
            "Part_1B": "nonadmitted",
            "Part_2": "incomplete",
            "Part_2A": "incomplete_dta",
            "Part_3": "new_periods",
        })
    )
    .pivot_table(
        index=[
            "period",
            "provider",
            "treatment",
        ],
        columns=["rtt_part_type"],
        values=["total_all", "wait_lt_18", "wait_gte_18", "wait_sum", "wait_diff","wait_pct_lt_18"],
        aggfunc="first"
    )
    .reset_index()
)
# df_wide["wait_lt_18"] = df.groupby(["period", "provider", "treatment"])["wait_lt_18"].first().values
# df_wide["wait_gte_18"] = df.groupby(["period", "provider", "treatment"])["wait_gte_18"].first().values
# df_wide = df_wide.merge(
#     df.groupby(["period", "provider", "treatment"])[["wait_lt_18", "wait_gte_18"]].first().reset_index(),
#     on=["period", "provider", "treatment"]
# )
cols_to_drop = [col for col in df_wide.columns if
                col[0] in ["wait_lt_18", "wait_gte_18", "wait_sum", "wait_diff", "wait_pct_lt_18"] and col[1] != "incomplete"]
df_wide = df_wide.drop(columns=cols_to_drop)

# Rename to flatten
df_wide.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in df_wide.columns.values]

df_wide = df_wide.rename(columns={
    'total_all_admitted': 'admitted',
    'total_all_incomplete': 'incomplete',
    'total_all_incomplete_dta': 'incomplete_dta',
    'total_all_new_periods': 'new_periods',
    'total_all_nonadmitted': "nonadmitted",
    'wait_diff_incomplete': "wait_diff",
    'wait_gte_18_incomplete': "wait_gte_18",
    'wait_lt_18_incomplete': "wait_lt_18",
    'wait_sum_incomplete': "wait_sum",
    'wait_pct_lt_18_incomplete': "wait_pct_lt_18"
})

print(df_wide.columns)

df_wide.query("provider == 'RAJ' and treatment == 'C_999' and period == '2025-08'")


Index(['period', 'provider', 'treatment', 'admitted', 'incomplete',
       'incomplete_dta', 'new_periods', 'nonadmitted', 'wait_diff',
       'wait_gte_18', 'wait_lt_18', 'wait_pct_lt_18', 'wait_sum'],
      dtype='object')


,period,provider,treatment,admitted,incomplete,incomplete_dta,new_periods,nonadmitted,wait_diff,wait_gte_18,wait_lt_18,wait_pct_lt_18,wait_sum
216420,2025-08,RAJ,C_999,3812.0,176763.0,19465.0,30748.0,18107.0,0.0,87362.0,89401.0,0.505768,176763.0


In [15]:

# Nan for treated metrics can be interpreted as 0, which is probably not valid generally
cols_to_fill: list[str] = ['admitted', 'nonadmitted', 'incomplete', 'new_periods']
df_wide[cols_to_fill] = df_wide[cols_to_fill].fillna(0).astype('int64')

df_wide["incomplete_prev"] = df_wide.groupby(["provider", "treatment"], as_index=False)['incomplete'].shift(1)
df_wide["admitted_prev"] = df_wide.groupby(["provider", "treatment"], as_index=False)['admitted'].shift(1)
df_wide["new_periods_prev"] = df_wide.groupby(["provider", "treatment"], as_index=False)['new_periods'].shift(1)
df_wide["nonadmitted_prev"] = df_wide.groupby(["provider", "treatment"], as_index=False)['nonadmitted'].shift(1)

df_wide["incomplete_diff"] = df_wide["incomplete"] - df_wide["incomplete_prev"]
df_wide["incomplete_expected"] = df_wide["incomplete_prev"] + df_wide["new_periods"] - df_wide["nonadmitted"] - df_wide[
    "admitted"]
df_wide["treated"] = df_wide["nonadmitted"] + df_wide["admitted"]
df_wide["treated_prev"] = df_wide["nonadmitted_prev"] + df_wide["admitted_prev"]

df_wide["incomplete_expected"] = (
        df_wide["incomplete_prev"]
        + df_wide["new_periods"]
        - df_wide["nonadmitted"]
        - df_wide["admitted"]
)

df_wide["untreated"] = df_wide["incomplete"] - df_wide["incomplete_expected"]

df_wide.query("provider == 'RAJ' and period == '2025-08'")


,period,provider,treatment,admitted,incomplete,incomplete_dta,new_periods,nonadmitted,wait_diff,wait_gte_18,...,wait_sum,incomplete_prev,admitted_prev,new_periods_prev,nonadmitted_prev,incomplete_diff,incomplete_expected,treated,treated_prev,untreated
216402,2025-08,RAJ,C_100,522,9390,1956.0,1745,1017,0.0,4886.0,...,9390.0,9282.0,534.0,2063.0,1073.0,108.0,9488.0,1539,1607.0,-98.0
216403,2025-08,RAJ,C_101,276,10178,1422.0,1760,1077,0.0,4804.0,...,10178.0,10263.0,300.0,2145.0,1122.0,-85.0,10670.0,1353,1422.0,-492.0
216404,2025-08,RAJ,C_110,359,18316,3581.0,2070,767,0.0,11242.0,...,18316.0,18014.0,399.0,2083.0,912.0,302.0,18958.0,1126,1311.0,-642.0
216405,2025-08,RAJ,C_120,152,15754,1014.0,1678,1446,0.0,10007.0,...,15754.0,15902.0,165.0,1783.0,1651.0,-148.0,15982.0,1598,1816.0,-228.0
216406,2025-08,RAJ,C_130,387,13434,1856.0,3165,1662,0.0,5554.0,...,13434.0,13060.0,475.0,3744.0,3021.0,374.0,14176.0,2049,3496.0,-742.0
216407,2025-08,RAJ,C_140,79,6232,571.0,628,553,0.0,4048.0,...,6232.0,5725.0,87.0,837.0,716.0,507.0,5721.0,632,803.0,511.0
216408,2025-08,RAJ,C_150,0,98,1.0,4,1,0.0,61.0,...,98.0,92.0,0.0,3.0,5.0,6.0,95.0,1,5.0,3.0
216409,2025-08,RAJ,C_160,535,6054,2984.0,946,182,0.0,2937.0,...,6054.0,5848.0,590.0,1046.0,228.0,206.0,6077.0,717,818.0,-23.0
216410,2025-08,RAJ,C_170,0,0,NaN,5,0,NaN,NaN,...,NaN,0.0,1.0,4.0,3.0,0.0,5.0,0,4.0,-5.0
216411,2025-08,RAJ,C_300,15,2519,60.0,484,132,0.0,868.0,...,2519.0,2503.0,18.0,777.0,169.0,16.0,2840.0,147,187.0,-321.0


In [ ]:

from scripts.excel_parsing_olde.parse_nhs_data import load_data_to_database2


def create_consolidated_table(conn) -> None:
    """Create the metrics table with synthetic columns."""
    cursor = conn.cursor()
    cursor.execute("""
                   CREATE TABLE IF NOT EXISTS consolidated
                   (
                       period              TEXT NOT NULL,
                       provider            TEXT NOT NULL,
                       treatment           TEXT NOT NULL,

                       -- Core metrics
                       incomplete          INTEGER,
                       incomplete_dta      INTEGER,
                       admitted            INTEGER,
                       new_periods         INTEGER,
                       nonadmitted         INTEGER,

                       -- Synthetic previous period columns
                       incomplete_prev     INTEGER,
                       incomplete_diff     INTEGER,
                       incomplete_expected INTEGER,
                       treated             INTEGER,
                       wait_gte_18         INTEGER,
                       wait_lt_18          INTEGER,
                       wait_pct_lt_18      INTEGER,
                       -- the residual (incomplete_t - incomplete_{t-1} + activty)
                       untreated           INTEGER,
                       -- integrity checking columns
                       wait_diff           INTEGER,
                       wait_sum            INTEGER,
                       admitted_prev       INTEGER,
                       new_periods_prev    INTEGER,
                       nonadmitted_prev    INTEGER,
                       treated_prev        INTEGER,

                       PRIMARY KEY (period, provider, treatment),
                       FOREIGN KEY (provider) REFERENCES providers (provider_code)
                   )
                   """)


connection = conn.raw_connection()
create_consolidated_table(connection)


In [ ]:

load_data_to_database2(df_wide, "consolidated", connection)
